Merge Data

In [10]:
import sys
print(sys.executable)

import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq



c:\Users\missm\OneDrive\Documents\Capstone_Project_Strickland\CapstoneProject_LearningPatterns\LP\Scripts\python.exe


In [ ]:
# Load Cleaned Datasets
kt1 = pd.read_parquet("KT1_cleaned_sample.parquet")
kt2 = pd.read_parquet("KT2_cleaned_sample.parquet")
kt3 = pd.read_parquet("KT3_cleaned_sample.parquet")
kt4 = pd.read_parquet("KT4_cleaned_sample.parquet")

print(kt1.shape, kt2.shape, kt3.shape, kt4.shape)


(1402362, 6) (6025501, 7) (9820184, 7) (15270789, 9)


In [3]:
# Standardize each tag to avoid future confusion
kt1["dataset"] = "KT1"
kt2["dataset"] = "KT2"
kt3["dataset"] = "KT3"
kt4["dataset"] = "KT4"


In [11]:
# Ensure common columns across all datsets
    # Make sure each has 'timestamp', 'user_id'
for df, tag in zip([kt1, kt2, kt3, kt4], ["KT1","KT2","KT3","KT4"]):
    df["dataset"] = tag
    if "timestamp" in df:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    else:
        df["timestamp"] = pd.NaT
    # ensure merge keys exist
    for col in ["question_id","elapsed_time","user_answer","action_type"]:
        if col not in df.columns:
            df[col] = pd.NA
   
    
    

# Add dummy columns where needed so schemas match (optional step for concat)
for col in ['question_id', 'elapsed_time', 'user_answer']:
    for df in [kt2, kt3, kt4]:
        if col not in df.columns:
            df[col] = pd.NA


In [13]:
# Merge datasets into unified log
full_log = pd.concat([kt1, kt2, kt3, kt4], ignore_index=True)
# keep only the columns you actually need downstream:
full_log = full_log[[
    "user_id","timestamp","dataset",
    "question_id","elapsed_time","user_answer","action_type"
]]

full_log.sort_values(by=["user_id", "timestamp"], inplace=True)
full_log.reset_index(drop=True, inplace=True)

#Downcast dtypes: shrink memory footprint
# Convert question_id to a smaller integer, answers to categories, etc.
full_log = full_log.astype({
    # pandas nullable float supports <NA>
    "elapsed_time":   "Float32",
    # categorical keys
    "question_id":    "category",
    "user_answer":    "category",
    "dataset":        "category",
    "action_type":    "category",
})


#Write to paraquet
full_log.to_parquet("EdNet_full_log.parquet", index=False)


In [15]:
#Merge with questions.csv to analyze correctness
questions = (pd.read_csv(r"C:\Users\missm\OneDrive\Documents\Capstone_Project_Strickland\CapstoneProject_LearningPatterns\questions.csv", usecols=["question_id","correct_answer"])
      .astype({"question_id":"category","correct_answer":"category"}))

#Chunked merge to handle larrge dataset
pf = pq.ParquetFile("EdNet_full_log.parquet")
writer = None
schema = None

for batch in pf.iter_batches(batch_size=1_000_000):
    # 1) back → pandas
    df_chunk = pa.Table.from_batches([batch]).to_pandas()

    # 2) re‐apply your dtype casts
    df_chunk = df_chunk.astype({
        "elapsed_time": "Float32",
        "question_id":  "category",
        "user_answer":  "category",
        "dataset":      "category",
        "action_type":  "category",
    })

    # 3) merge & compute is_correct
    merged = df_chunk.merge(questions, on="question_id", how="left")
    merged["is_correct"] = merged["user_answer"] == merged["correct_answer"]

    if writer is None:
        # first chunk → infer & lock in the schema
        table  = pa.Table.from_pandas(merged, preserve_index=False)
        schema = table.schema
        writer = pq.ParquetWriter("EdNet_full_with_correct.parquet", schema)
        writer.write_table(table)
    else:
        # subsequent chunks → enforce the same schema
        # (1) reorder columns to match schema.names
        merged = merged[schema.names]
        # (2) convert to Arrow Table using that schema
        table = pa.Table.from_pandas(merged, schema=schema, preserve_index=False)
        writer.write_table(table)

if writer:
    writer.close()


In [17]:
# Compute Student Features

# 1️⃣ Read in the merged log
df = pd.read_parquet("EdNet_full_with_correct.parquet")

# 2️⃣ Group & aggregate
student_features = (
    df.groupby("user_id")
      .agg(
          total_questions    = ("question_id",   "count"),
          avg_elapsed_time   = ("elapsed_time",  "mean"),
          accuracy_rate      = ("is_correct",    "mean"),
          num_responses      = ("user_answer",   "count"),
          num_actions        = ("action_type",   "count"),
          first_timestamp    = ("timestamp",     "min"),
          last_timestamp     = ("timestamp",     "max"),
      )
)

# 3️⃣ Compute active days (will be NaN if timestamps missing)
student_features["active_days"] = (
    student_features["last_timestamp"]
    - student_features["first_timestamp"]
).dt.days

# 4️⃣ Fill only the numeric holes (leave timestamps alone)
student_features = student_features.fillna({
    "total_questions": 0,
    "avg_elapsed_time": 0.0,
    "accuracy_rate":    0.0,
    "num_responses":    0,
    "num_actions":      0,
    "active_days":      0,
})

# 5️⃣ Downcast feature dtypes for extra space savings
student_features = student_features.astype({
    "total_questions":  "Int32",    # nullable int
    "avg_elapsed_time": "Float32",  # nullable float
    "accuracy_rate":    "Float32",
    "num_responses":    "Int32",
    "num_actions":      "Int32",
    "active_days":      "Int32",
    # timestamps can stay as-is (datetime64[ns])
})

# 6️⃣ Write out your final Parquet
student_features.to_parquet(
    "student_behavior_features.parquet",
    index=True  # keeps the user_id index
)

print("Wrote student_behavior_features.parquet with shape", student_features.shape)


Wrote student_behavior_features.parquet with shape (9746, 8)


In [18]:
student_features = pd.read_parquet("student_behavior_features.parquet")
